In [1]:
import pandas as pd
import numpy as np

DATA_DIR = r'D:\Tseng\圍棋\singletrials_timewindows'
BANDS = ['Alpha', 'Beta', 'Delta', 'Gamma', 'Theta']
PARIETAL_CH = ['P3', 'P4', 'P7', 'P8', 'Pz', 'CP1', 'CP2', 'CP5', 'CP6']

# T_19_5s 在所有 CSV 中為空（trailing comma），排除，使用 38 個時間點
time_vals = [x / 10 for x in range(5, 195, 5)]   # 0.5, 1.0, ..., 19.0
TIME_COLS = []
for v in time_vals:
    int_part = int(v)
    dec_part = round((v - int_part) * 10)
    TIME_COLS.append(f'T_{int_part}_{dec_part}s')

print(f'有效時間點欄位數: {len(TIME_COLS)}')   # 應為 38
print('前3:', TIME_COLS[:3])
print('後3:', TIME_COLS[-3:])

有效時間點欄位數: 38
前3: ['T_0_5s', 'T_1_0s', 'T_1_5s']
後3: ['T_18_0s', 'T_18_5s', 'T_19_0s']


In [2]:
# ── 載入並驗證五個頻帶的 CSV ──
dfs = {}
for band in BANDS:
    path = f'{DATA_DIR}\\Original_{band}.csv'
    df = pd.read_csv(path)
    dfs[band] = df
    print(f'{band}: {df.shape}  Group 分布: {df["Group"].value_counts().to_dict()}')

Alpha: (161152, 51)  Group 分布: {'amateur': 78720, 'master': 59424, 'prof': 23008}
Beta: (161152, 51)  Group 分布: {'amateur': 78720, 'master': 59424, 'prof': 23008}
Delta: (161152, 51)  Group 分布: {'amateur': 78720, 'master': 59424, 'prof': 23008}
Gamma: (161152, 51)  Group 分布: {'amateur': 78720, 'master': 59424, 'prof': 23008}
Theta: (161152, 51)  Group 分布: {'amateur': 78720, 'master': 59424, 'prof': 23008}


In [3]:
# ── RT1_ms 品質檢查（只看 master + Parietal）──
print('=== RT1_ms 品質檢查 ===')
for band in BANDS:
    df_m = dfs[band][(dfs[band]['Group'] == 'master') &
                     (dfs[band]['Channel'].isin(PARIETAL_CH))].copy()

    rt = df_m['RT1_ms']
    # 強制轉數值，無法轉的變 NaN
    rt_num = pd.to_numeric(rt, errors='coerce')

    n_total  = len(rt_num)
    n_nan    = rt_num.isna().sum()
    n_bad    = (rt != rt_num).sum()   # 原本不是 NaN 但轉換後變 NaN → 非數值字串

    print(f'[{band}] rows={n_total}  NaN={n_nan}  非數值字串={n_bad}')
    if n_nan > 0:
        print('  NaN 的 SubjectID/Session/Epoch:')
        bad_rows = df_m[rt_num.isna()][['SubjectID','Session','Epoch','RT1_ms']]
        print(bad_rows.drop_duplicates().to_string(index=False))

=== RT1_ms 品質檢查 ===
[Alpha] rows=16713  NaN=0  非數值字串=0
[Beta] rows=16713  NaN=0  非數值字串=0
[Delta] rows=16713  NaN=0  非數值字串=0
[Gamma] rows=16713  NaN=0  非數值字串=0
[Theta] rows=16713  NaN=0  非數值字串=0


In [4]:
# ── 前處理函式 ──
def preprocess_band(df, band_name):
    # Step 1: 只取 master
    df_m = df[df['Group'] == 'master'].copy()

    # Step 2: 只取 Parietal channels
    df_p = df_m[df_m['Channel'].isin(PARIETAL_CH)].copy()

    # Step 3: RT1_ms 轉數值，過濾掉 NaN 的 trial
    df_p['RT1_ms'] = pd.to_numeric(df_p['RT1_ms'], errors='coerce')
    n_before = df_p[['SubjectID','Session','Epoch']].drop_duplicates().shape[0]
    # 找出有 NaN RT 的 trial key
    bad_trials = df_p[df_p['RT1_ms'].isna()][['SubjectID','Session','Epoch']].drop_duplicates()
    if len(bad_trials) > 0:
        print(f'[{band_name}] 移除 {len(bad_trials)} 個 RT1_ms 無效的 trial')
        df_p = df_p.merge(bad_trials.assign(_drop=True),
                          on=['SubjectID','Session','Epoch'], how='left')
        df_p = df_p[df_p['_drop'].isna()].drop(columns='_drop')
    n_after = df_p[['SubjectID','Session','Epoch']].drop_duplicates().shape[0]
    print(f'[{band_name}] master trials: {n_before} → 清理後 {n_after}')

    # Step 4: groupby trial → channel 平均（38 個時間點）
    grp = df_p.groupby(['SubjectID', 'Session', 'Epoch'])
    avg = grp[TIME_COLS].mean()          # (n_trials, 38)
    rt  = grp['RT1_ms'].first()

    # Step 5: sum=1 標準化
    row_sums = avg.sum(axis=1)
    avg_norm = avg.div(row_sums, axis=0)

    # 加頻帶前綴
    avg_norm.columns = [f'{band_name}_{c}' for c in TIME_COLS]

    return avg_norm, rt


# ── 對五個頻帶執行 ──
band_features = {}
rt_series = None

for band in BANDS:
    feat, rt = preprocess_band(dfs[band], band)
    band_features[band] = feat
    if rt_series is None:
        rt_series = rt
    else:
        if not feat.index.equals(rt_series.index):
            raise ValueError(f'{band} 的 trial index 與其他頻帶不一致，請檢查資料！')
    print()

[Alpha] master trials: 1857 → 清理後 1857

[Beta] master trials: 1857 → 清理後 1857

[Delta] master trials: 1857 → 清理後 1857

[Gamma] master trials: 1857 → 清理後 1857

[Theta] master trials: 1857 → 清理後 1857



In [5]:
# ── concat 五個頻帶 → 190 維特徵 ──
X = pd.concat([band_features[b] for b in BANDS], axis=1)
y = rt_series.rename('RT1_ms')

print('X shape:', X.shape)   # 預期 (n_trials, 190)
print('y shape:', y.shape)
print('X NaN 數:', X.isna().sum().sum())
print('y NaN 數:', y.isna().sum())
print('\nRT1_ms 統計:')
print(y.describe())

X shape: (1857, 190)
y shape: (1857,)
X NaN 數: 0
y NaN 數: 0

RT1_ms 統計:
count     1857.000000
mean     25069.782445
std      15982.820925
min       1232.000000
25%      11806.000000
50%      22128.000000
75%      34282.000000
max      58962.000000
Name: RT1_ms, dtype: float64


In [6]:
# ── 儲存 ──
import os
out_dir = r'd:\Tseng\game_eeg\RT1_ms_predict'

df_out = X.copy()
df_out['RT1_ms'] = y
df_out = df_out.reset_index()

out_path = os.path.join(out_dir, 'master_parietal_190d.csv')
df_out.to_csv(out_path, index=False)
print(f'已儲存: {out_path}')
print(f'Shape: {df_out.shape}')
print(df_out.iloc[:3, :6])   # 預覽前幾欄

已儲存: d:\Tseng\game_eeg\RT1_ms_predict\master_parietal_190d.csv
Shape: (1857, 194)
  SubjectID Session  Epoch  Alpha_T_0_5s  Alpha_T_1_0s  Alpha_T_1_5s
0    sub-52    ss01      1      0.014533      0.025483      0.021811
1    sub-52    ss01      2      0.017716      0.011836      0.018507
2    sub-52    ss01      3      0.017205      0.018018      0.031943
